### Topic 3: Why Qdrant Specifically?

### Step 1: We Needed a Vector Database

Many vector databases are available, such as FAISS, Chroma, Qdrant, Pinecone, and Weaviate.

Instead of asking **"Which is the best?"**, we asked:

**"Which one best fits our project?"**

Our requirements were: **Easy to learn, Runs locally, Supports metadata filtering, Easy to move to production, Open source, Production ready.**

Qdrant satisfied all these requirements.

---

### Step 2: Easy to Learn

For learning, Qdrant runs directly inside Python.

```python
QdrantClient(":memory:")
```

No Docker, server, or cloud account is required.

---

### Step 3: Easy to Move to Production

Development:

```python
QdrantClient(":memory:")
```

Production:

```python
QdrantClient(url="http://localhost:6333")
```

Only the client configuration changes. Collections, upsert, search, and filtering remain the same.

---

### Step 4: Metadata Filtering

Suppose the knowledge base contains **Banking FAQs, Insurance FAQs, HR Policies, and Product Manuals**.

For a Fixed Deposit query, Qdrant can search only documents where **Product = Fixed Deposit** and **Language = English**, improving retrieval quality.

---

### Step 5: Persistent Storage

`:memory:` is ideal for learning but loses data after restart. Production uses Docker, which stores vectors permanently.

---

### Step 6: Open Source & Production Ready

Qdrant provides **Persistent storage, Metadata filtering, REST API, gRPC API, Docker deployment, and Cloud deployment.**

---

### Example

A banking chatbot stores one million FAQ chunks containing **Embedding, Original text, Product, Language, and Source PDF**.

For a Fixed Deposit query, Qdrant searches only the relevant documents instead of the entire knowledge base.

---

### Common Mistakes

- Using `:memory:` in production.
- Forgetting Docker volumes.
- Changing the embedding model without recreating the collection.
- Ignoring metadata filtering.

---

### Key Takeaways

- Choose the database that fits your project.
- `:memory:` is for learning.
- Docker provides persistence.
- Metadata filtering improves retrieval quality.
- Moving to production requires changing only the client configuration.

### Persistent Storage

**Definition**

Persistent Storage means **data is permanently saved on disk and remains available even after the application, server, or computer restarts.**

**Example**

Without persistent storage:

1. Store 1 million document embeddings.
2. Restart the application.
3. All embeddings are lost.

With persistent storage:

1. Store 1 million document embeddings.
2. Restart the application.
3. All embeddings are still available.

**Qdrant Example**

- `QdrantClient(":memory:")` stores data only in RAM. Restarting Python deletes all vectors.
- Qdrant running with Docker and a persistent volume stores data on disk, so vectors remain available after restarts.

---

### Lead-Level Interview Questions

### Q1. Why did you choose Qdrant over other vector databases?

**Answer**

We selected Qdrant because it matched our project requirements. It is easy to learn, supports metadata filtering, provides persistent storage, is open source, and can move from development to production with minimal code changes.

---

### Q2. Why is `QdrantClient(":memory:")` useful?

**Answer**

It allows developers to build and test applications without Docker or a server. It is ideal for learning and local development but should not be used in production because all data is lost after a restart.

---

### Q3. What is persistent storage?

**Answer**

Persistent storage saves vectors on disk, so they remain available even after the application or server restarts. It is essential for production systems.

---

### Q4. Why is metadata filtering important?

**Answer**

Metadata filtering limits the search to relevant documents, such as **Product = Fixed Deposit** or **Language = English**. This improves both retrieval speed and result quality.

---

### Q5. What changes when moving from development to production?

**Answer**

Only the client configuration changes. The collection creation, upsert, search, and filtering logic remain the same.

---

### Q6. What happens if you change the embedding model?

**Answer**

A different embedding model may produce vectors with a different dimension. The existing collection cannot store vectors of a new dimension, so the collection must be recreated and the documents re-embedded.

---

### Q7. What are common mistakes when using Qdrant?

**Answer**

Using `:memory:` in production, forgetting Docker volumes, changing embedding models without rebuilding the collection, and not using metadata filtering.

---

In [1]:
"""
qdrant_setup.py
-----------------
Qdrant setup and connection verification.

Uses :memory: mode -- no Docker required, no network download,
starts instantly inside the Python process. Full Qdrant behavior
(collections, points, payloads, filtering, search) works identically
in :memory: mode and Docker mode.

The ONLY difference between modes is one line in get_client():
  Learning / no Docker : QdrantClient(":memory:")
  Local Docker         : QdrantClient(url="http://localhost:6333")
  Qdrant Cloud         : QdrantClient(url="https://...", api_key="...")

When you want persistence:
  docker run -p 6333:6333 -v <your_path>/qdrant_storage:/qdrant/storage qdrant/qdrant
  Then swap get_client() to QdrantClient(url="http://localhost:6333").
  Nothing else in this file or any other file needs to change.

Install: pip install qdrant-client
"""

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # paraphrase-multilingual-MiniLM-L12-v2 output dimension


def get_client() -> QdrantClient:
    """Returns a Qdrant client.

    Currently uses :memory: mode -- no Docker needed, zero setup,
    data is lost when the Python process restarts.

    To switch to persistent local Docker, replace with:
        return QdrantClient(url="http://localhost:6333")

    To switch to Qdrant Cloud, replace with:
        return QdrantClient(url="https://your-cluster.qdrant.io", api_key="YOUR_KEY")
    """
    return QdrantClient(":memory:")


def create_collection(client: QdrantClient, recreate: bool = False) -> None:
    """Creates the collection if it does not already exist.

    recreate=True drops and rebuilds the collection -- use this when
    changing embedding models or chunk schemas. Never use on a live
    collection with real data.
    """
    # check what collections already exist so we do not crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    if COLLECTION_NAME in existing:
        if recreate:
            # deliberate teardown -- only when we know we want a clean slate
            client.delete_collection(COLLECTION_NAME)
            print(f"Deleted existing collection '{COLLECTION_NAME}'")
        else:
            print(f"Collection '{COLLECTION_NAME}' already exists -- skipping creation")
            return

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,           # every vector in this collection must be 384 floats
            distance=Distance.COSINE,   # cosine similarity -- right choice for normalized vectors
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}' (vector_size={VECTOR_SIZE})")


def verify_connection(client: QdrantClient) -> None:
    """Quick sanity check -- prints collection info if everything is working."""
    info = client.get_collection(COLLECTION_NAME)
    print(f"Collection   : {COLLECTION_NAME}")
    print(f"Vector size  : {info.config.params.vectors.size}")
    print(f"Points count : {info.points_count}")
    print(f"Status       : {info.status}")


# ----------------------------------------------------------------------
# Run setup
# ----------------------------------------------------------------------

# get a client -- :memory: mode, no Docker needed
client = get_client()

# create the collection
create_collection(client, recreate=False)

# confirm everything is working
verify_connection(client)

Created collection 'fd_knowledge_base' (vector_size=384)
Collection   : fd_knowledge_base
Vector size  : 384
Points count : 0
Status       : green
